In [1]:
import os
import urllib.request

with open("C:\\Users\\shash\\Code\\Training_LLM\\archive\\the-verdict.txt", "r") as file:
    text_data = file.read()

In [2]:
print(text_data[:500])  # Print the first 500 characters to verify the content

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it'


In [3]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(f"Number of Characters in the text: {len(text_data)}")
print(f"Number of Tokens in the text: {len(tokenizer.encode(text_data))}")

Number of Characters in the text: 20479
Number of Tokens in the text: 5145


In [4]:
from torch.utils.data import Dataset, DataLoader
import torch
class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
    

def create_dataloader(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    dataset = GPTDataset(text, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [5]:
### Training Configuration ###
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12, 
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [6]:
train_ratio = 0.90
split_index = int(len(text_data) * train_ratio)
train_text = text_data[:split_index]
val_text = text_data[split_index:]

torch.manual_seed(123)

train_loader = create_dataloader(
    train_text,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)


val_loader = create_dataloader(
    val_text,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [7]:
# ### Basic Sanity Check ###
# if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
#     print("Warning: The training dataset is too small for the specified context length. Consider reducing the context length or increasing the dataset size.")

# if total_tokens * (1 - train_ratio) < GPT_CONFIG_124M["context_length"]:
#     print("Warning: The validation dataset is too small for the specified context length. Consider reducing the context length or increasing the dataset size.")

In [8]:
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")

print("Sample Batch from Training Loader:")
for x, y in train_loader:
    print("Input IDs Shape:", x.shape)
    print("Target IDs Shape:", y.shape)
    break

print("Sample Batch from Validation Loader:")
for x, y in val_loader:
    print("Input IDs Shape:", x.shape)
    print("Target IDs Shape:", y.shape)
    break


Number of training batches: 9
Number of validation batches: 1
Sample Batch from Training Loader:
Input IDs Shape: torch.Size([2, 256])
Target IDs Shape: torch.Size([2, 256])
Sample Batch from Validation Loader:
Input IDs Shape: torch.Size([2, 256])
Target IDs Shape: torch.Size([2, 256])


In [9]:
from model import GPTModel
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (token_embedding): Embedding(50257, 768)
  (position_embedding): Embedding(256, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer_block): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layer): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
       

In [10]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

def calc_loss_loader(dataloader, model, device, num_batches=None):
    total_loss = 0.0
    if len(dataloader) == 0:
        return float('nan')
    elif num_batches is None:
        num_batches = len(dataloader)
    else:
        num_batches = min(num_batches, len(dataloader))
    
    for i, (input_batch, target_batch) in enumerate(dataloader):
        if i >= num_batches:
            break
        batch_loss = calc_loss_batch(input_batch, target_batch, model, device)
        total_loss += batch_loss.item()
    return total_loss / num_batches

In [13]:
%%time
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

CPU times: total: 1min 21s
Wall time: 23.9 s


In [14]:
print(f"Initial Training Loss: {train_loss:.4f}")
print(f"Initial Validation Loss: {val_loss:.4f}")

Initial Training Loss: 10.9751
Initial Validation Loss: 10.9701
